## Data generation

### Import and setup

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Set a random seed for reproducible results

np.random.seed(42)

### Simulating Funnel Drop-offs & Submissions

In [ ]:
# Define sample sizes for a balanced A/B test

n_A = 10000
n_B = 10000

total_n = n_A + n_B

print(f"Simulating data for {total_n} total users (A: {n_A}, B: {n_B})...")

In [ ]:
# Create base DataFrame

df_A = pd.DataFrame({'group': ['A'] * n_A})
df_B = pd.DataFrame({'group': ['B'] * n_B})

In [ ]:
# --- Version A Funnel Logic ---
# High progression rates through basic steps

df_A['reached_step_1'] = 1  # Starts at Step 1
df_A['reached_step_2'] = np.random.binomial(1, 0.92, n_A)
df_A['reached_step_3'] = df_A['reached_step_2'] * np.random.binomial(1, 0.88, n_A)
df_A['submitted'] = df_A['reached_step_3'] * np.random.binomial(1, 0.95, n_A)

In [ ]:
# --- Version B Funnel Logic ---
# Step 0 acts as a heavy friction filter

df_B['reached_step_0'] = 1  # Starts at Step 0
df_B['reached_step_1'] = np.random.binomial(1, 0.82, n_B) # Drop-off due to disclaimer agreement
df_B['reached_step_2'] = df_B['reached_step_1'] * np.random.binomial(1, 0.90, n_B)
df_B['reached_step_3'] = df_B['reached_step_2'] * np.random.binomial(1, 0.86, n_B)
df_B['submitted'] = df_B['reached_step_3'] * np.random.binomial(1, 0.96, n_B) # Slightly higher conditional submission at the very end

In [ ]:
# Combine the datasets

df = pd.concat([df_A, df_B], ignore_index=True).fillna(0).astype({'reached_step_0': int, 'reached_step_1': int, 'reached_step_2': int, 'reached_step_3': int, 'submitted': int})

### Simulating Estimated Costs & Hotline Calls

In [ ]:
# ---Estimated Costs (Only generated for users who submitted a claim) ---

# Version A: Log-normal distribution skewed toward higher inflated values (Mean ~4500)

costs_A = np.random.lognormal(mean=8.2, sigma=0.6, size=total_n)

# Version B: Shifted lower to reflect lower fraudulent/inflated values (Mean ~3300, ~1200 lower)

costs_B = np.random.lognormal(mean=7.9, sigma=0.6, size=total_n)

In [ ]:
df['estimated_cost'] = np.where(df['group'] == 'A', costs_A, costs_B)

df.loc[df['submitted'] == 0, 'estimated_cost'] = np.nan # No cost if they didn't submit

In [ ]:
# --- 4. Hotline Calls (Simulated via a Poisson distribution for all visitors) ---

df['hotline_calls'] = np.random.poisson(lam = 0.15, size = total_n)

### KPI Aggregation and Validation

In [ ]:
# 1 & 2. Submission & Drop-off Analysis

print("=== FUNNEL PERFORMANCE & DROP-OFFS ===")
funnel_metrics = df.groupby('group')[['reached_step_0', 'reached_step_1', 'reached_step_2', 'reached_step_3', 'submitted']].mean()
print(funnel_metrics.round(4))

sub_rate_A = funnel_metrics.loc['A', 'submitted']
sub_rate_B = funnel_metrics.loc['B', 'submitted']
print(f"\nObserved Submission Rate Effect: {sub_rate_B - sub_rate_A:.2%} (Target: -8.0%)")

# Calculate explicit drop-off rate per step

print("\n=== ABSOLUTE DROP-OFF RATE PER VISIT FROM PREVIOUS STEP ===")
drop_0_to_1 = df.groupby('group').apply(lambda x: 1 - (x['reached_step_1'].sum() / x['reached_step_0'].sum() if 'reached_step_0' in x and x['reached_step_0'].sum() > 0 else 1))
drop_1_to_2 = df.groupby('group').apply(lambda x: 1 - (x['reached_step_2'].sum() / x['reached_step_1'].sum()))
drop_2_to_3 = df.groupby('group').apply(lambda x: 1 - (x['reached_step_3'].sum() / x['reached_step_2'].sum()))
drop_3_to_sub = df.groupby('group').apply(lambda x: 1 - (x['submitted'].sum() / x['reached_step_3'].sum()))

# Calculate average macro drop-off per group across the core operational workflow steps (1->2, 2->3, 3->sub)

macro_drop_A = np.mean([drop_1_to_2['A'], drop_2_to_3['A'], drop_3_to_sub['A']])
macro_drop_B = np.mean([drop_0_to_1['B'], drop_1_to_2['B'], drop_2_to_3['B'], drop_3_to_sub['B']])
print(f"Average Macro Step Drop-off Rate: Group A = {macro_drop_A:.2%}, Group B = {macro_drop_B:.2%}")
print(f"Observed Treatment Effect on Drop-offs: {macro_drop_B - macro_drop_A:.2%} (Target: +5.0%)")

print("\n" + "="*40 + "\n")

# 3. Estimated Costs Analysis

print("=== ESTIMATED CLAIMS COSTS ===")
cost_summary = df.groupby('group')['estimated_cost'].agg(['mean', 'median'])
print(cost_summary.round(2))
mean_diff = cost_summary.loc['B', 'mean'] - cost_summary.loc['A', 'mean']
print(f"Observed Mean Cost Difference: {mean_diff:.2f} EUR (Target: -1200.00 EUR)")

print("\n" + "="*40 + "\n")

# 4. Hotline Calls Analysis

print("=== HOTLINE CALLS ===")
hotline_summary = df.groupby('group')['hotline_calls'].agg(['mean', 'median'])
print(hotline_summary.round(4))